# Data quality

## Critical fields
- Row id
- Customer id
- Sales
- Dates
- Discount

In [14]:
import pandas as pd

df = pd.read_csv("data/raw/samplesuperstore.csv")

print(f"Rows:{len(df):,}")
df.head()

Rows:10,194


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,1/3/2023,1/7/2023,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512
1,2,US-2023-112326,1/4/2023,1/8/2023,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,3,US-2023-112326,1/4/2023,1/8/2023,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
3,4,US-2023-112326,1/4/2023,1/8/2023,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
4,5,US-2023-141817,1/5/2023,1/12/2023,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840


In [15]:

quality = pd.DataFrame({
    "Column": df.columns,
    "Dtype": df.dtypes.astype(str),
    "Nulls": df.isna().sum().values,
    "Unique": df.nunique().values,
})

quality

,Column,Dtype,Nulls,Unique
Row ID,Row ID,int64,0,10194
Order ID,Order ID,str,0,5111
Order Date,Order Date,str,0,1242
Ship Date,Ship Date,str,0,1338
Ship Mode,Ship Mode,str,0,4
Customer ID,Customer ID,str,0,804
Customer Name,Customer Name,str,0,800
Segment,Segment,str,0,3
Country/Region,Country/Region,str,0,2
City,City,str,0,542


In [16]:
rules = [
    ("DQ001", "Uniqueness","row_id_unique", 1.00),
    ("DQ002", "Consistency","delivery_after_order", 1.00),
    ("DQ003", "Validity","sales_not_negative",1.00),
    ("DQ004", "Validity","quantity_bigger_than_zero", 1.00),
    ("DQ005", "Validity","discount_valid", 1.00)
]

In [ ]:
# Convert date column to datetime 
df["Order Date"] = pd.to_datetime(df["Order Date"],format = '%m/%d/%Y')
df["Ship Date"] = pd.to_datetime(df["Ship Date"],format = '%m/%d/%Y')

In [18]:
results = []


results.append({
    "Rule": "DQ01",
    "Dimension": "Uniqueness",
    "Metric": df["Row ID"].nunique() / len(df),
    "Target": 1.00
})

results.append({
    "Rule": "DQ02",
    "Dimension": "Consistency",
    "Metric": (df["Order Date"] <= df["Ship Date"]).mean(),
    "Target": 1.00
})

results.append({
    "Rule": "DQ03",
    "Dimension": "Validity",
    "Metric": (df["Sales"] >= 0 ).mean(),
    "Target": 1.00
})

results.append({
    "Rule": "DQ04",
    "Dimension": "Validity",
    "Metric": (df["Quantity"] > 0 ).mean(),
    "Target": 1.00
})

results.append({
    "Rule": "DQ05",
    "Dimension": "Validity",
    "Metric": ((df["Discount"] >= 0) & (df["Discount"] < 1)).mean(),
    "Target": 1.00
})

In [19]:
results_df = pd.DataFrame(results)

results_df["Status"] = (
    results_df["Metric"] >= results_df["Target"]
).map({
    False: "Fail",
    True: "Pass"
})

results_df

,Rule,Dimension,Metric,Target,Status
0,DQ01,Uniqueness,1.0,1.0,Pass
1,DQ02,Consistency,1.0,1.0,Pass
2,DQ03,Validity,1.0,1.0,Pass
3,DQ04,Validity,1.0,1.0,Pass
4,DQ05,Validity,1.0,1.0,Pass


In [22]:
df.to_csv("./data/processed/processed_samplesuperstore.csv", index=False)